<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-11 14:21:59
=================== PORTFOLIO QUALITY ====================
qualified stocks: 84
with latest results: 84
still star stocks: 44
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.55 C
CAGR/XIRR %:     1.61%
Est FTT Value:   2.74 C
Future Upside:   1.19 C (76.64%)
=================== PERFORMANCE ==========================
Current PnL:             -40.31 L (-21.0%)
Total Booked PnL:        49.23 L (33.02%)
Booked + Current PnL:    8.91 L (5.98%)
CY Booked PnL:           6.36 L (4.1%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               1.86 L (1.21%)
CY Investment:            1.92 C
Reserve:                  2.04 K
=================== P&L DETAIL ============================
Total Profit:             2.38 L
Total Loss:               -42.69 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
75,TTKPRESTIG,-2.10,-23.07,30.11,0.09,23343.0,-23251.0,77526.0,770.00,-6.45,66.0,M-SC,1.97,250.0,-1.00,0.50,39.54,OX40N,NTT,DURABLES
12,BOSCH-HCIL,2.56,-22.66,29.29,-0.01,28950.0,-28965.0,98840.0,2282.00,774.55,52.0,M-SC,19.79,231.0,-1.00,0.64,68.24,OX40N,BTT,AC
50,M&M,3.86,6.70,10.11,17.48,32478.0,20170.0,321245.0,3762.88,-6.96,80.0,X-LC,6.61,25.0,0.62,2.07,24.22,X40N,ATH,AUTO
65,SIS,0.69,-9.62,34.16,21.26,34281.0,-10677.0,100355.0,528.00,93.09,51.0,H-SC,7.29,168.0,-0.31,0.65,49.21,OX40N,NTT,MISC
52,MASFIN,1.67,-17.84,47.58,21.26,38303.0,-17478.0,80502.0,396.02,-36.34,53.0,H-SC,14.65,167.0,-0.46,0.52,9.68,XR,ATH,FINANCE


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
38,HYUNDAI,2.88,11.00,28.19,42.29,88733.0,31195.0,314766.0,2782.73,16.23,68.0,X-LC,3.88,29.0,0.35,2.03,29.57,X40N,ATH,AUTO
47,JIOFIN,0.31,-18.87,67.74,36.09,220681.0,-75768.0,325777.0,387.00,-37.82,65.0,X-LC,9.75,34.0,-0.34,2.10,12.82,XY24,BTT,FINANCE
51,MARUTI,1.83,-3.12,30.04,25.99,98869.0,-10585.0,329125.0,17120.06,-9.72,76.0,X-LC,11.50,33.0,-0.11,2.12,17.86,XR,ATH,AUTO
4,ASIANPAINT,1.43,-7.17,34.47,24.83,114048.0,-25554.0,330863.0,3422.40,-19.19,66.0,X-LC,4.95,30.0,-0.22,2.14,21.31,X40,ATH,PAINTS
44,INFY,-0.50,-16.93,71.73,42.66,238866.0,-67848.0,333007.0,1972.00,-39.69,57.0,X-LC,7.91,3.0,-0.28,2.15,19.21,X40,NTT,IT


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
69,TANLA,6.60,-21.86,175.37,115.17,371027.0,-59193.0,211568.0,1903.93,NaN,79.0,NaN,17.18,147.0,-0.16,1.37,87.03,AR,ATH,IT
79,VALIANTORG,6.58,-24.26,129.94,74.15,161013.0,-39692.0,123913.0,838.00,NaN,40.0,NaN,39.68,145.0,-0.25,0.80,84.40,XR,NTT,CHEMICALS
63,SFL,4.98,-42.39,119.81,26.63,180984.0,-111159.0,151059.0,1287.00,NaN,52.0,NaN,7.92,242.0,-0.61,0.98,26.07,XY24,NTT,MISC
42,INDIGOPNTS,4.91,-31.81,46.67,0.02,55556.0,-55519.0,119040.0,1408.00,-34.92,45.0,M-SC,7.74,222.0,-1.00,0.77,35.63,OX40N,NTT,PAINTS
59,RELAXO,4.28,-60.74,287.93,52.31,164097.0,-88168.0,56992.0,1176.00,NaN,58.0,NaN,13.15,92.0,-0.54,0.37,28.86,X40N,NTT,FOOTWEAR


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
75,TTKPRESTIG,-2.10,-23.07,30.11,0.09,23343.0,-23251.0,77526.0,770.00,-6.45,66.0,M-SC,1.97,250.0,-1.00,0.50,39.54,OX40N,NTT,DURABLES
1,ABBOTINDIA,-1.76,-12.07,31.21,15.37,49081.0,-21595.0,157260.0,34390.26,-28.41,51.0,X-MC,0.93,58.0,-0.44,1.02,6.53,X40,ATH,PHARMA
53,NESTLEIND,-1.52,7.25,20.49,29.22,68613.0,22624.0,334862.0,1540.00,-19.03,52.0,X-LC,8.58,20.0,0.33,2.16,10.07,XY25,ATH,FMCG
13,BSOFT,-1.45,-40.63,147.91,47.20,123523.0,-57141.0,83512.0,811.90,NaN,58.0,NaN,10.28,172.0,-0.46,0.54,23.25,XR,ATH,IT
76,UNITDSPR,-1.17,3.99,16.13,20.77,39274.0,9339.0,243483.0,1644.00,-11.27,58.0,X-MC,2.70,64.0,0.24,1.57,17.04,X40N,NTT,BREWERIES


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,1.43,58.32,30.40,106.44,71060.0,86106.0,233750.0,1629.94,15.40,51.0,M-SC,24.26,220.0,1.21,1.51,77.11,OX40N,ATH,CABLES
65,SIS,0.69,-9.62,34.16,21.26,34281.0,-10677.0,100355.0,528.00,93.09,51.0,H-SC,7.29,168.0,-0.31,0.65,49.21,OX40N,NTT,MISC
12,BOSCH-HCIL,2.56,-22.66,29.29,-0.01,28950.0,-28965.0,98840.0,2282.00,774.55,52.0,M-SC,19.79,231.0,-1.00,0.64,68.24,OX40N,BTT,AC
75,TTKPRESTIG,-2.10,-23.07,30.11,0.09,23343.0,-23251.0,77526.0,770.00,-6.45,66.0,M-SC,1.97,250.0,-1.00,0.50,39.54,OX40N,NTT,DURABLES
48,KANSAINER,2.61,-31.17,64.86,13.47,120385.0,-84060.0,185607.0,340.00,-27.90,66.0,H-SC,6.11,162.0,-0.70,1.20,31.19,XY24,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,1.43,58.32,30.4,106.44,71060.0,86106.0,233750.0,1629.94,15.4,51.0,M-SC,24.26,220.0,1.21,1.51,77.11,OX40N,ATH,CABLES


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,1.43,58.32,30.40,106.44,71060.0,86106.0,233750.0,1629.94,15.40,51.0,M-SC,24.26,220.0,1.21,1.51,77.11,OX40N,ATH,CABLES
66,STARHEALTH,1.93,8.14,36.17,47.26,109962.0,22891.0,304014.0,761.00,-4.77,56.0,H-SC,4.36,173.0,0.21,1.96,31.45,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,1.43,58.32,30.40,106.44,71060.0,86106.0,233750.0,1629.94,15.40,51.0,M-SC,24.26,220.0,1.21,1.51,77.11,OX40N,ATH,CABLES
83,WIPRO,-0.82,-9.30,112.80,93.01,256029.0,-23264.0,226976.0,420.00,-23.61,66.0,M-LC,7.39,98.0,-0.09,1.47,25.88,XR,NTT,IT
69,TANLA,6.60,-21.86,175.37,115.17,371027.0,-59193.0,211568.0,1903.93,NaN,79.0,NaN,17.18,147.0,-0.16,1.37,87.03,AR,ATH,IT
79,VALIANTORG,6.58,-24.26,129.94,74.15,161013.0,-39692.0,123913.0,838.00,NaN,40.0,NaN,39.68,145.0,-0.25,0.80,84.40,XR,NTT,CHEMICALS
41,INDIAMART,0.61,-33.29,201.20,100.92,165531.0,-41064.0,82272.0,4675.60,NaN,45.0,NaN,16.78,140.0,-0.25,0.53,4.38,AR,ATH,MISC


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
6,AWL,0.35,-42.51,190.52,67.02,390252.0,-151474.0,204835.0,485.00,-57.56,39.0,X-SC,8.27,81.0,-0.39,1.32,1.89,XY24,NTT,FMCG
37,HONAUT,2.04,-20.36,89.46,50.88,137388.0,-39263.0,153575.0,58192.80,-37.74,44.0,X-SC,7.51,90.0,-0.29,0.99,17.01,X40N,ATH,ELECTRICAL
11,BERGEPAINT,0.65,-12.60,43.38,25.32,97722.0,-32480.0,225269.0,680.00,-23.95,51.0,X-MC,2.28,68.0,-0.33,1.45,18.13,XY24,NTT,PAINTS
1,ABBOTINDIA,-1.76,-12.07,31.21,15.37,49081.0,-21595.0,157260.0,34390.26,-28.41,51.0,X-MC,0.93,58.0,-0.44,1.02,6.53,X40,ATH,PHARMA
53,NESTLEIND,-1.52,7.25,20.49,29.22,68613.0,22624.0,334862.0,1540.00,-19.03,52.0,X-LC,8.58,20.0,0.33,2.16,10.07,XY25,ATH,FMCG


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
17,CIPLA,0.87,6.44,16.39,23.89,52451.0,19361.0,320016.0,1648.14,20.68,67.0,X-LC,0.69,18.0,0.37,2.07,19.87,X40N,ATH,PHARMA
1,ABBOTINDIA,-1.76,-12.07,31.21,15.37,49081.0,-21595.0,157260.0,34390.26,-28.41,51.0,X-MC,0.93,58.0,-0.44,1.02,6.53,X40,ATH,PHARMA
54,PAGEIND,-0.03,-5.14,40.05,32.85,101963.0,-13789.0,254590.0,50936.07,-29.48,56.0,X-MC,2.27,55.0,-0.14,1.64,21.57,X40,ATH,APPARELS
11,BERGEPAINT,0.65,-12.60,43.38,25.32,97722.0,-32480.0,225269.0,680.00,-23.95,51.0,X-MC,2.28,68.0,-0.33,1.45,18.13,XY24,NTT,PAINTS
76,UNITDSPR,-1.17,3.99,16.13,20.77,39274.0,9339.0,243483.0,1644.00,-11.27,58.0,X-MC,2.70,64.0,0.24,1.57,17.04,X40N,NTT,BREWERIES


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
6,AWL,0.35,-42.51,190.52,67.02,390252.0,-151474.0,204835.0,485.00,-57.56,39.0,X-SC,8.27,81.0,-0.39,1.32,1.89,XY24,NTT,FMCG
1,ABBOTINDIA,-1.76,-12.07,31.21,15.37,49081.0,-21595.0,157260.0,34390.26,-28.41,51.0,X-MC,0.93,58.0,-0.44,1.02,6.53,X40,ATH,PHARMA
20,DABUR,1.52,-25.52,92.94,43.70,171717.0,-63307.0,184761.0,735.00,-35.67,54.0,X-MC,12.39,70.0,-0.37,1.19,7.46,XY24,BTT,FMCG
26,GILLETTE,0.97,-12.29,46.72,28.69,104041.0,-31206.0,222690.0,10891.01,-36.71,58.0,X-SC,6.14,83.0,-0.30,1.44,7.60,X40,ATH,FMCG
28,GODREJCP,2.88,-20.39,65.93,32.10,130976.0,-50878.0,198659.0,1452.16,-21.90,59.0,X-MC,16.14,32.0,-0.39,1.28,7.64,X40N,ATH,FMCG


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
55,PGHH,0.67,-46.16,143.33,31.02,164784.0,-98552.0,114968.0,17484.41,-60.42,57.0,X-SC,23.47,57.0,-0.60,0.74,8.23,X40,ATH,FMCG
2,ACC,0.09,-44.68,216.02,74.81,320401.0,-119808.0,148320.0,3906.00,-44.95,62.0,X-SC,14.34,85.0,-0.37,0.96,9.88,XY24,BTT,CEMENT
37,HONAUT,2.04,-20.36,89.46,50.88,137388.0,-39263.0,153575.0,58192.80,-37.74,44.0,X-SC,7.51,90.0,-0.29,0.99,17.01,X40N,ATH,ELECTRICAL
1,ABBOTINDIA,-1.76,-12.07,31.21,15.37,49081.0,-21595.0,157260.0,34390.26,-28.41,51.0,X-MC,0.93,58.0,-0.44,1.02,6.53,X40,ATH,PHARMA
81,VOLTAS,2.05,-12.98,66.85,45.20,111546.0,-24882.0,166860.0,1856.00,19.81,60.0,X-MC,19.43,79.0,-0.22,1.08,17.56,XY25,NTT,AC


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
46,ITC,2.13,-18.92,58.74,28.70,208408.0,-82797.0,354798.0,452.00,-26.68,67.0,X-LC,7.31,1.0,-0.40,2.29,11.67,X40,NTT,FMCG
71,TCS,-0.90,-26.11,71.54,26.75,257988.0,-127455.0,360620.0,4208.34,-40.29,57.0,X-LC,7.48,2.0,-0.49,2.33,25.08,X40,ATH,IT
44,INFY,-0.50,-16.93,71.73,42.66,238866.0,-67848.0,333007.0,1972.00,-39.69,57.0,X-LC,7.91,3.0,-0.28,2.15,19.21,X40,NTT,IT
73,TMPV,1.58,-25.29,100.23,49.60,357103.0,-120588.0,356284.0,600.00,-85.99,65.0,X-LC,16.52,4.0,-0.34,2.30,14.72,XY24,NTT,AUTO
33,HCLTECH,-0.66,-4.36,31.44,25.70,108541.0,-15741.0,345232.0,1837.07,-8.82,56.0,X-LC,8.64,8.0,-0.15,2.23,37.74,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
75,TTKPRESTIG,-2.10,-23.07,30.11,0.09,23343.0,-23251.0,77526.0,770.00,-6.45,66.0,M-SC,1.97,250.0,-1.00,0.50,39.54,OX40N,NTT,DURABLES
12,BOSCH-HCIL,2.56,-22.66,29.29,-0.01,28950.0,-28965.0,98840.0,2282.00,774.55,52.0,M-SC,19.79,231.0,-1.00,0.64,68.24,OX40N,BTT,AC
79,VALIANTORG,6.58,-24.26,129.94,74.15,161013.0,-39692.0,123913.0,838.00,NaN,40.0,NaN,39.68,145.0,-0.25,0.80,84.40,XR,NTT,CHEMICALS
69,TANLA,6.60,-21.86,175.37,115.17,371027.0,-59193.0,211568.0,1903.93,NaN,79.0,NaN,17.18,147.0,-0.16,1.37,87.03,AR,ATH,IT
27,GLAXO,0.43,9.26,24.19,35.69,52670.0,18452.0,217736.0,3380.16,-18.03,62.0,X-MC,8.62,56.0,0.35,1.41,29.41,X40,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
69,TANLA,6.60,-21.86,175.37,115.17,371027.0,-59193.0,211568.0,1903.93,NaN,79.0,NaN,17.18,147.0,-0.16,1.37,87.03,AR,ATH,IT
83,WIPRO,-0.82,-9.30,112.80,93.01,256029.0,-23264.0,226976.0,420.00,-23.61,66.0,M-LC,7.39,98.0,-0.09,1.47,25.88,XR,NTT,IT
50,M&M,3.86,6.70,10.11,17.48,32478.0,20170.0,321245.0,3762.88,-6.96,80.0,X-LC,6.61,25.0,0.62,2.07,24.22,X40N,ATH,AUTO
13,BSOFT,-1.45,-40.63,147.91,47.20,123523.0,-57141.0,83512.0,811.90,NaN,58.0,NaN,10.28,172.0,-0.46,0.54,23.25,XR,ATH,IT
61,REPCOHOME,1.41,-15.40,124.38,89.82,301472.0,-44125.0,242380.0,880.00,NaN,77.0,NaN,12.41,161.0,-0.15,1.56,29.16,XY24,NTT,FINANCE


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.62
1,25,51.49
2,50,82.29


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    43.09
SC    20.33
MC    19.79
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.22
XY24     22.92
X40N     19.22
XR       10.01
XY25      9.91
OX40N     6.75
AR        3.26
SR        0.75
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    41.62
X-MC    17.35
H-SC     8.21
X-SC     7.00
M-SC     5.12
M-LC     1.47
M-MC     1.11
H-MC     1.00
L-MC     0.33
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,18.24,-24.62,69.58
IT,12.77,-29.91,97.60
FINANCE,11.03,-19.23,69.98
AUTO,9.42,-9.47,49.68
PAINTS,5.56,-22.96,45.04
MISC,5.53,-44.02,101.91
INSURANCE,5.33,-0.05,37.65
BANKS,5.17,-12.03,68.21
PHARMA,4.50,2.33,22.19


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,3942521.0,19
X40,2118037.0,16
XR,1554063.0,13
X40N,1455327.0,12
AR,1004349.0,6
XY25,756404.0,6
OX40N,745307.0,10
SR,298091.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2889740.0,19
X-MC,1547732.0,13
X-SC,1190248.0,6
H-SC,880232.0,8
M-SC,368935.0,6
H-MC,259229.0,1
M-LC,256029.0,1
M-MC,74107.0,1
L-MC,51089.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1278958.0      8
X-SC       XY24       710653.0      2
X-LC       XY24       577784.0      2
           X40N       552770.0      5
H-SC       XY24       537578.0      3
X-MC       X40N       527690.0      4
           X40        449665.0      5
X-LC       XY25       381359.0      3
X-MC       XY25       300938.0      2
           XY24       269439.0      2
X-SC       X40        268825.0      2
H-MC       XY24       259229.0      1
M-LC       XR         256029.0      1
H-SC       OX40N      236787.0      3
X-SC       X40N       210770.0      2
M-SC       OX40N      178909.0      4
           XY24       129435.0      1
H-SC       XR         105867.0      2
X-LC       XR          98869.0      1
M-MC       XY25        74107.0      1
M-SC       XR          60591.0      1
L-MC       XR          51089.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 41.0 seconds
